# Download and Preview Market Data

This notebook runs the first stage of the project: download raw equity history and option-chain data from Yahoo Finance, save the raw CSV files, and inspect the returned tables.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "src" / "bs_pricer").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bs_pricer.config import MarketConfig, list_snapshots
from bs_pricer.pipeline import run_pipeline
from bs_pricer.plotting import plot_volatility_smile, plot_volatility_surface

PROJECT_ROOT

In [ ]:
config = MarketConfig(
    ticker="TSLA",
    tag="",  # optional free-form label to distinguish pulls, e.g. "0dte", "leaps"
    start_date="2023-01-01",
    end_date=None,
    raw_data_dir=PROJECT_ROOT / "data" / "raw",
    processed_data_dir=PROJECT_ROOT / "data" / "processed",
)

config

## Volatility Comparison Setup

For SPY, this notebook downloads equity history from 2023-01-01 onward. Later, realized volatility can be compared with option maturity buckets using common trading-day windows: 21 days for roughly 1 month, 63 days for roughly 3 months, and 126 days for roughly 6 months.

## 1. Download Live Data

This cell fetches a fresh market snapshot from Yahoo Finance and overwrites the saved CSVs in `data/raw/` and `data/processed/`. Option quotes move continuously, so every re-run of this cell changes every downstream number and plot.

Run it once per session. If you just want to re-inspect or re-plot the last saved snapshot, skip straight to the next section instead of re-running this cell.

In [ ]:
result = run_pipeline(config)

print(f"Saved snapshot: {result['snapshot_id']}")
(
    result["equity_path"],
    result["options_path"],
    result["equity_processed_path"],
    result["options_processed_path"],
    result["options_with_iv_path"],
)

## 2. Load Saved Snapshot for Analysis

Every download in section 1 is saved under its own `snapshot_id` (`{ticker}_{tag}_{timestamp}`), so different pulls -- different tickers, tags, or times -- never overwrite each other. Pick which one to inspect below, then everything from here on reads back that snapshot's CSVs instead of talking to Yahoo Finance again.

In [ ]:
available_snapshots = list_snapshots(config)
available_snapshots

In [ ]:
snapshot_id = available_snapshots[0]  # <- set to a different entry above to inspect an older/other snapshot
snapshot_id

In [ ]:
equity_history = pd.read_csv(
    config.raw_data_dir / f"{snapshot_id}_equity_history.csv", index_col=0
)
equity_history.index = pd.to_datetime(equity_history.index, utc=True)

option_chains = pd.read_csv(config.raw_data_dir / f"{snapshot_id}_option_chains.csv")

equity_clean = pd.read_csv(config.processed_data_dir / f"{snapshot_id}_equity_clean.csv")
options_clean = pd.read_csv(config.processed_data_dir / f"{snapshot_id}_options_clean.csv")
options_with_iv = pd.read_csv(
    config.processed_data_dir / f"{snapshot_id}_options_with_iv.csv"
)

equity_history.shape, option_chains.shape, equity_clean.shape, options_clean.shape, options_with_iv.shape

In [ ]:
options_clean.head()

In [ ]:
options_with_iv.head()

In [ ]:
price_col = "Adj Close" if "Adj Close" in equity_history.columns else "Close"

ax = equity_history[price_col].plot(
    figsize=(12, 5),
    title=f"{config.ticker} {price_col} Price",
)
ax.set_xlabel("Date")
ax.set_ylabel("Price")

In [ ]:
equity_history.tail()

In [ ]:
option_chains.head()

In [ ]:
option_chains.columns.tolist()

In [ ]:
option_chains.groupby(["expiry", "option_type"]).size().head(20)

In [ ]:
option_chains[["contractSymbol", "expiry", "option_type", "strike", "lastPrice", "bid", "ask", "volume", "openInterest", "impliedVolatility"]].head(20)

## Cleaned Data Preview

In [ ]:
equity_clean.head()

In [ ]:
options_clean.head()

In [ ]:
options_clean[["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "days_to_expiry", "time_to_expiry"]].head(20)

## Implied Volatility Preview

In [ ]:
options_with_iv.head()

In [ ]:
options_with_iv["solved_iv"].describe()

In [ ]:
options_with_iv[["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "time_to_expiry", "solved_iv"]].head(20)

In [ ]:
options_with_iv[
    ["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "time_to_expiry", "solved_iv"]
].dropna(subset=["solved_iv"]).sort_values(["expiry", "option_type", "strike"]).head(50)

In [ ]:
options_with_iv[
    ["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "time_to_expiry", "solved_iv"]
].dropna(subset=["solved_iv"]).sample(20, random_state=1)

## Volatility Smile and Surface

In [ ]:
plot_volatility_smile(options_with_iv, option_type="call", x="moneyness")

In [ ]:
plot_volatility_surface(options_with_iv, option_type="call", x="moneyness")